# Notebook 10 — MCA commercial-vs-Medicare reconciliation

**Purpose.** Close the Day-13 open item: was MCA's commercial-to-Medicare multiple **2.6×** or **~3.9×**?

**Self-contained.** This notebook pulls its own data and runs end-to-end — just **Kernel → Restart & Run All**. No dependency on Notebook 9's kernel state.

**Cell legend**
- **▶ RUN IN ENV** — needs the local DuckDB files.
- **✓ TESTED** — DB-independent logic, exercised against a synthetic frame before shipping.


## Step 0 — Setup  ▶ RUN IN ENV

Same project-root walk-up and `data/raw/*.duckdb` glob as Notebook 9, so the hospital labels match.

In [1]:
import sys, os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Walk UP to the folder that holds src/queries.py (this notebook lives in a subdir,
# so cwd is NOT the project root -- that was the old ModuleNotFoundError).
def find_project_root(start=None, marker="src/queries.py", max_up=6):
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents][:max_up + 1]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f"Could not find {marker} above {p}")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

from src.queries import add_lob_v4, query_procedure_rates_agg

TARGET_CODE = "73721"
SETTING     = "outpatient"

DB_GLOB  = str(PROJECT_ROOT / "data" / "raw" / "*.duckdb")
db_paths = sorted(glob.glob(DB_GLOB))
assert db_paths, f"No DuckDBs matched {DB_GLOB!r} -- fix DB_GLOB"

LABEL_RULES = [
    ("baylor",       "Baylor"),
    ("medical_city", "MCA"),        # Medical City Alliance
    ("methodist",    "Methodist"),
    ("parkland",     "Parkland"),
    ("presbyterian", "THP"),        # Texas Health Presbyterian
    ("texas_health", "THP"),
]
def _label(stem):
    s = stem.lower()
    for kw, lab in LABEL_RULES:
        if kw in s:
            return lab
    return stem

DB_PATHS = {_label(Path(p).stem): p for p in db_paths}
assert len(DB_PATHS) == len(db_paths), "Two DBs mapped to the same label -- fix LABEL_RULES"
print(f"{len(DB_PATHS)} DBs:")
for label, p in DB_PATHS.items():
    print(f"  {label:10s} <- {Path(p).name}")

PROJECT_ROOT: C:\Users\kedha\Documents\dfw-hospital-pricing
5 DBs:
  Baylor     <- baylor_university_medical_center-69947_parsed.duckdb
  MCA        <- medical_city_alliance_hospital-77912_parsed.duckdb
  Methodist  <- methodist_dallas_medical_center-6000b_parsed.duckdb
  Parkland   <- parkland_health-6e88d_parsed.duckdb
  THP        <- texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb


## Step 1 — Pull the negotiated rates  ▶ RUN IN ENV

`query_procedure_rates_agg` per DB, tagged with the hospital label. Same pull as Notebook 9.

In [2]:
frames = []
for label, path in DB_PATHS.items():
    con = duckdb.connect(path, read_only=True)
    try:
        df = query_procedure_rates_agg(con, TARGET_CODE, SETTING)
    finally:
        con.close()
    df.insert(0, "hospital", label)
    frames.append(df)
    print(f"{label:10s} {len(df):4d} rows")

rates = pd.concat(frames, ignore_index=True)
print("\nTOTAL rate rows:", len(rates))

Baylor       45 rows
MCA          46 rows
Methodist    82 rows
Parkland     65 rows
THP          40 rows

TOTAL rate rows: 278


## Step 2 — Classify (v4) and keep dollar-priced rows  ▶ RUN IN ENV

Carries the two Notebook-9 decisions:
- **Decision 29** — keep all billing classes (`billing_class_normalized` is a hospital-level tagging artifact here, not a professional/facility clinical split).
- **Decision 30** — dollar-priced rows form the ladder; percentage-of-charges rows excluded.

In [3]:
rates  = add_lob_v4(rates)
dollar = rates[rates["dollar_rate"].notna()].copy()
print(f"dollar-priced rows: {len(dollar)}  (of {len(rates)} total)")

dollar-priced rows: 239  (of 278 total)


## Step R — The reconciliation  ✓ TESTED

Compute everything from `dollar`. First show what MCA actually files (the evidence), then the numbers.

In [4]:
# --- Evidence: what MCA actually files for 73721 (dollar-priced) ---
mca = dollar[dollar.hospital == "MCA"]
print("MCA dollar-priced LOB coverage:")
print(mca["lob"].value_counts().to_string())
print("  commercial_specialty rows:", int((mca.lob == "commercial_specialty").sum()))
print("  medicare rows            :", int(mca.lob.isin(["medicare_traditional", "medicare_advantage"]).sum()), "\n")

# --- The reconciliation ---
_MEDICARE_LOBS = ["medicare_traditional", "medicare_advantage"]
mca_commercial = dollar[(dollar.hospital == "MCA") & (dollar.lob == "commercial")]["dollar_rate"].median()
medicare_rung  = dollar[dollar.lob.isin(_MEDICARE_LOBS)].groupby("hospital")["dollar_rate"].median()
mca_has_own_medicare = "MCA" in medicare_rung.index

peer_ma                = medicare_rung.drop("MCA", errors="ignore")
peer_ma_median_all     = peer_ma.median()
peer_ma_median_cluster = peer_ma[peer_ma < 500].median()
ratio_all, ratio_cluster = mca_commercial / peer_ma_median_all, mca_commercial / peer_ma_median_cluster
implied_denom_26 = mca_commercial / 2.6

print(f"MCA commercial rung (numerator) : ${mca_commercial:,.2f}  (single bucket; no commercial_specialty)")
print(f"MCA has its OWN Medicare rung?   : {mca_has_own_medicare}  (so the rung is an honest NaN)\n")
print("Peer Medicare-Advantage rungs:")
print(peer_ma.round(2).to_string())
print(f"  peer-MA median, all peers     : ${peer_ma_median_all:,.2f}")
print(f"  peer-MA median, <$500 cluster : ${peer_ma_median_cluster:,.2f}  (Parkland out; median barely moves)\n")
print(f"3.9x check (all peers)          : {mca_commercial:,.2f} / {peer_ma_median_all:,.2f} = {ratio_all:,.2f}x")
print(f"3.9x check (cluster)            : {mca_commercial:,.2f} / {peer_ma_median_cluster:,.2f} = {ratio_cluster:,.2f}x")
print(f"2.6x would require denominator  : {mca_commercial:,.2f} / 2.6 = ${implied_denom_26:,.2f}")
print(f"  -> no peer rung sits near ${implied_denom_26:,.0f}; 2.6x not reproducible -> retire it")

MCA dollar-priced LOB coverage:
lob
commercial            15
medicaid_star          6
medicaid_other         5
medicaid_chip          5
medicaid_star_plus     2
aca_exchange           1
  commercial_specialty rows: 0
  medicare rows            : 0 

MCA commercial rung (numerator) : $906.53  (single bucket; no commercial_specialty)
MCA has its OWN Medicare rung?   : False  (so the rung is an honest NaN)

Peer Medicare-Advantage rungs:
hospital
Baylor        235.30
Methodist     232.47
Parkland     1274.07
THP           226.38
  peer-MA median, all peers     : $233.88
  peer-MA median, <$500 cluster : $232.47  (Parkland out; median barely moves)

3.9x check (all peers)          : 906.53 / 233.88 = 3.88x
3.9x check (cluster)            : 906.53 / 232.47 = 3.90x
2.6x would require denominator  : 906.53 / 2.6 = $348.67
  -> no peer rung sits near $349; 2.6x not reproducible -> retire it


## Findings — open item closed

- **The numerator is locked.** MCA's commercial rung is a single `commercial` bucket at **$906.53** — no `commercial_specialty` rows, and it's a plain median, so the Decision-31 family-pooling change could not have moved it.
- **MCA has no native Medicare rung.** It files no Medicare(-MA) rate for 73721, so the ladder leaves it an honest NaN. Any "MCA commercial vs Medicare" figure is *peer-borrowed*.
- **3.9× is the defensible figure:** $906.53 ÷ peer-hospital MA median (~$232–234, robust to whether Parkland is included) ≈ **3.9×**.
- **2.6× is retired.** It would need a ~$349 denominator, and no peer Medicare rung sits anywhere near $349 (the cluster is ~$226–235; Parkland's $1,274 is the lone outlier). It was a stale Day-13 hand-calc against a denominator that no longer exists in the data.

**Carry into any writeup:** present 3.9× only as "MCA commercial ≈ 3.9× *peer* Medicare Advantage," never as MCA's own rate. The post-ready cash multiples (cash ≈ 9× commercial, ≈ 40× Medicaid) are unaffected — they don't touch this borrowed proxy.

In [5]:
# === Step R — MCA "2.6x vs 3.9x" reconciliation (scratch, Day 15) ===========
# Closes the Day-13 open item. Self-contained: computes from `dollar` (Step 2),
# so it doesn't depend on the Step 4b overlay cell having run this session.
# Conclusion: numerator is locked, MCA has no native Medicare rung, 3.9x is a
# *peer-borrowed* contextual figure, and 2.6x can't be rebuilt from current data.

assert "dollar" in globals(), "Run the notebook through Step 2 (Cell 9) first -- `dollar` isn't defined yet."
_MEDICARE_LOBS = ["medicare_traditional", "medicare_advantage"]

mca_commercial = dollar[(dollar.hospital == "MCA") & (dollar.lob == "commercial")]["dollar_rate"].median()
medicare_rung  = (dollar[dollar.lob.isin(_MEDICARE_LOBS)]
                  .groupby("hospital")["dollar_rate"].median())
mca_has_own_medicare = "MCA" in medicare_rung.index

peer_ma                = medicare_rung.drop("MCA", errors="ignore")
peer_ma_median_all     = peer_ma.median()
peer_ma_median_cluster = peer_ma[peer_ma < 500].median()
ratio_all, ratio_cluster = mca_commercial/peer_ma_median_all, mca_commercial/peer_ma_median_cluster
implied_denom_26 = mca_commercial / 2.6

print(f"MCA commercial rung (numerator) : ${mca_commercial:,.2f}  (single 15-row bucket; no commercial_specialty)")
print(f"MCA has its OWN Medicare rung?   : {mca_has_own_medicare}  (NaN in the ladder = honest absence)\n")
print("Peer Medicare-Advantage rungs:")
print(peer_ma.round(2).to_string())
print(f"  peer-MA median, all peers     : ${peer_ma_median_all:,.2f}")
print(f"  peer-MA median, <$500 cluster : ${peer_ma_median_cluster:,.2f}  (Parkland out; median barely moves)\n")
print(f"3.9x check (all peers)          : {mca_commercial:,.2f} / {peer_ma_median_all:,.2f} = {ratio_all:,.2f}x")
print(f"3.9x check (cluster)            : {mca_commercial:,.2f} / {peer_ma_median_cluster:,.2f} = {ratio_cluster:,.2f}x")
print(f"2.6x would require denominator  : {mca_commercial:,.2f} / 2.6 = ${implied_denom_26:,.2f}")
print(f"  -> no peer rung sits near ${implied_denom_26:,.0f}; 2.6x not reproducible -> retire it")

MCA commercial rung (numerator) : $906.53  (single 15-row bucket; no commercial_specialty)
MCA has its OWN Medicare rung?   : False  (NaN in the ladder = honest absence)

Peer Medicare-Advantage rungs:
hospital
Baylor        235.30
Methodist     232.47
Parkland     1274.07
THP           226.38
  peer-MA median, all peers     : $233.88
  peer-MA median, <$500 cluster : $232.47  (Parkland out; median barely moves)

3.9x check (all peers)          : 906.53 / 233.88 = 3.88x
3.9x check (cluster)            : 906.53 / 232.47 = 3.90x
2.6x would require denominator  : 906.53 / 2.6 = $348.67
  -> no peer rung sits near $349; 2.6x not reproducible -> retire it
